In [22]:
from openai import OpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_sambanova import SambaNovaEmbeddings
import os
from langchain_ollama import OllamaEmbeddings

# --- LLM Backend Configuration ---
# Un‑comment one block at a time.

# (1) SambaNova
BACKEND = "sambanova"
CHAT_MODEL = "Llama-4-Maverick-17B-128E-Instruct"
BASE_URL = "https://ai.tejas.tacc.utexas.edu"
API_KEY = api_key_here
EMBEDDING_MODEL = "E5-Mistral-7B-Instruct"
os.environ["SAMBANOVA_API_BASE"] = BASE_URL
os.environ["SAMBANOVA_API_KEY"] = API_KEY


# (2) Ollama (OpenAI‑compatible API)
#BACKEND = "ollama"
#CHAT_MODEL = "llama3.2"  # or whatever model you run locally
#BASE_URL = "http://localhost:11434/v1"
#API_KEY = "ollama"  # any dummy string; Ollama ignores it
#EMBEDDING_MODEL = "nomic-embed-text"

# RAG parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
MAX_NEW_TOKENS = 512
NUM_K = 3

In [23]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


def llm_invoke(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_NEW_TOKENS,
        temperature=0.2,
    )
    return resp.choices[0].message.content

In [24]:
def main():
    # ---------- RAG: loading and chunking ---------- #
    loader = DirectoryLoader(
        "./docs/",
        glob="**/*.pdf",
        show_progress=False,
        loader_cls=PyPDFLoader,
        load_hidden=False,
    )
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    texts = text_splitter.split_documents(docs)

#    embeddings = OllamaEmbeddings(
#        model=EMBEDDING_MODEL,
#    )
#    embeddings = OpenAIEmbeddings(
#        model=EMBEDDING_MODEL,             
#    )

    embeddings = SambaNovaEmbeddings(
        model=EMBEDDING_MODEL
    )
    
    db = Chroma.from_documents(
        texts,
        embeddings,
        persist_directory="db_ce",
    )

    # ---------- Test Cases ---------- #
    messages = [
        "Describe Aurora's neural network architecture.",
        "What did the authors use as the training objective?",
    ]

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful AI assistant.
            Use the following retrieved context to help answer the question.
            Keep your answers concise. Two sentences maximum.

            Question: {question}

            Retrieved Context:
            {context}
            """
    )

    for question in messages:
        results = db.similarity_search(question, k=NUM_K)
        retrieved_context = "\n\n".join(result.page_content for result in results)

        formatted = prompt.format(question=question, context=retrieved_context)

        response = llm_invoke(formatted)

        print("\n ===================================================== \n")
        print(f"The Question:\n{question}\n")
        print("The Answer:")
        print(f"{response}\n")
        print("The Retrieved Context:")
        for result in results:
            print(f"{result.page_content}\n")
        print("\n ===================================================== \n")

In [25]:
main()



The Question:
Describe Aurora's neural network architecture.

The Answer:
Aurora's neural network architecture is based on a Multiscale 3D Swin Transformer U-Net backbone, which is a type of Vision Transformer that uses local self-attention operations within windows and a symmetric upsampling-downsampling structure. The backbone has a symmetric structure with three stages each, enabling multiscale processing through 3D Swin Transformer layers.

The Retrieved Context:
specifics on input processing, pressure-level aggregation and further 
encodings, see Supplementary Information Sections B.1 and B.4.
Multiscale 3D Swin Transformer U-Net backbone. The backbone 
of Aurora is a 3D Swin Transformer U-Net19,50, which serves as a neural 
simulator (see Fig. B1 Supplementary Information Section B.1). This 
architecture allows for efficient simulation of underlying physics at 
several scales. This architecture falls under the general family of Vision 
Transformers. However, unlike classical Vi